In [1]:
import os
import re
import asyncio
import aiohttp
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
import time

c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [5]:
# CHAPTER_URL = "https://demonicscans.org/title/Revenge-of-the-Iron%25252DBlooded-Sword-Hound/chapter/"
# CHAPTER_URL = "https://demonicscans.org/title/Murim-Login/chapter/"
# CHAPTER_URL = "https://demonicscans.org/title/Magic-Emperor/chapter/"
CHAPTER_URL = "https://demonicscans.org/title/Reborn-Rich/chapter/"

MAX_EMPTY_CHAPTERS = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    # "User-Agent":     "Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.5 Mobile/15E148 Safari/604.1",
    "Referer": "https://demonicscans.org/"
}

In [6]:
async def download_comic_images_async(base_chapter_url, chapter_start, chapter_max, max_concurrent=5):
    """Async download with concurrent chapters and images"""
    
    comic_name = unquote(unquote(base_chapter_url.split("/")[4]))
    comic_dir = os.path.join("downloads", comic_name)
    os.makedirs(comic_dir, exist_ok=True)
    
    panel_pattern = re.compile(r"/(\d+)\.(jpg|webp)$", re.IGNORECASE)
    empty_count = 0
    
    connector = aiohttp.TCPConnector(limit=max_concurrent)
    timeout = aiohttp.ClientTimeout(total=30)
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=HEADERS) as session:
        for i in range(chapter_start, chapter_max + 1):
            chapter_url = f"{base_chapter_url}{i}/1"
            print(f"Fetching chapter {i}...")
            
            try:
                async with session.get(chapter_url) as resp:
                    if resp.status in (404, 410):
                        print(f"Chapter {i} not found, stopping.")
                        break
                    if resp.status != 200:
                        print(f"Chapter {i}: Unexpected status {resp.status}, stopping.")
                        break
                    
                    html = await resp.text()
            except asyncio.TimeoutError:
                print(f"Chapter {i}: Timeout, stopping.")
                break
            except Exception as e:
                print(f"Chapter {i}: Error {e}, stopping.")
                break
            
            soup = BeautifulSoup(html, "html.parser")
            candidates = set()
            
            for img in soup.find_all("img"):
                for attr in ("src", "data-src", "data-lazy", "data-original"):
                    url = img.get(attr)
                    if not url:
                        continue
                    full_url = urljoin(chapter_url, url)
                    if panel_pattern.search(full_url):
                        candidates.add(full_url)
            
            if not candidates:
                empty_count += 1
                print(f"Chapter {i}: No images found.")
                if empty_count >= MAX_EMPTY_CHAPTERS:
                    print("Multiple empty chapters detected. Stopping.")
                    break
                continue
            else:
                empty_count = 0
            
            print(f"Chapter {i}: Found {len(candidates)} images, downloading...")
            
            chapter_dir = os.path.join(comic_dir, f"chapter_{i}")
            os.makedirs(chapter_dir, exist_ok=True)
            
            # Download all images concurrently for this chapter
            tasks = [
                download_image(session, url, chapter_dir, panel_pattern)
                for url in candidates
            ]
            await asyncio.gather(*tasks, return_exceptions=True)
    
    print("\nAll done.")

async def download_image(session, url, chapter_dir, panel_pattern):
    """Download a single image"""
    match = panel_pattern.search(url)
    if not match:
        return
    
    page, ext = match.groups()
    out_path = os.path.join(chapter_dir, f"{int(page):03}.{ext}")
    
    if os.path.exists(out_path):
        return  # Skip if already downloaded
    
    try:
        async with session.get(url) as resp:
            if resp.status == 200:
                with open(out_path, "wb") as f:
                    f.write(await resp.read())
    except Exception as e:
        print(f"Failed to download {url}: {e}")


In [7]:
await download_comic_images_async(CHAPTER_URL, 1, 210, max_concurrent=10)

Fetching chapter 1...
Chapter 1: Found 28 images, downloading...
Fetching chapter 2...
Chapter 2: Found 25 images, downloading...
Fetching chapter 3...
Chapter 3: Found 20 images, downloading...
Fetching chapter 4...
Chapter 4: Found 12 images, downloading...
Fetching chapter 5...
Chapter 5: Found 18 images, downloading...
Fetching chapter 6...
Chapter 6: Found 13 images, downloading...
Fetching chapter 7...
Chapter 7: Found 20 images, downloading...
Fetching chapter 8...
Chapter 8: Found 17 images, downloading...
Fetching chapter 9...
Chapter 9: Found 12 images, downloading...
Fetching chapter 10...
Chapter 10: Found 18 images, downloading...
Fetching chapter 11...
Chapter 11: Found 12 images, downloading...
Fetching chapter 12...
Chapter 12: Found 12 images, downloading...
Fetching chapter 13...
Chapter 13: Found 17 images, downloading...
Fetching chapter 14...
Chapter 14: Found 17 images, downloading...
Fetching chapter 15...
Chapter 15: Found 16 images, downloading...
Fetching chap